# Stage 11 — advanced model comparison

Compare a tuned gradient boosting model with the baseline results. The history experiment explicitly removes every `static_*` column because those are full-day aggregates and would leak future behavior into earlier hours if present in the saved CSV.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, ParameterGrid
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import precision_score,recall_score,f1_score,average_precision_score,roc_auc_score,confusion_matrix

BASE=Path('../outputs/model_ready/all_naive_12pct_200sims'); OUT=Path('../outputs/models/advanced_comparison'); OUT.mkdir(parents=True,exist_ok=True); SEED=42
history=pd.read_csv(BASE/'history_aware_sim_hour_features.csv',dtype={'sim':'string'},parse_dates=['hour_ts'])
print('history rows:',len(history),'static columns present:',[c for c in history.columns if c.startswith('static_')])

In [ ]:
target='fraud'; group='sim'; excluded={target,group,'hour_ts','known_fraud','static_fraud','record_id','event_key','src_number','dst_number','src_imei','dst_imei'}
excluded.update(c for c in history.columns if c.startswith('static_'))
columns=[c for c in history.select_dtypes(include='number').columns if c not in excluded]
X=history[columns].replace([np.inf,-np.inf],np.nan); y=history[target].astype(int); groups=history[group].astype(str)
first=GroupShuffleSplit(n_splits=1,test_size=.40,random_state=SEED); train,remain=next(first.split(X,y,groups))
second=GroupShuffleSplit(n_splits=1,test_size=.50,random_state=SEED+1); val_rel,test_rel=next(second.split(X.iloc[remain],y.iloc[remain],groups.iloc[remain])); val=remain[val_rel]; test=remain[test_rel]
print('features:',len(columns),'SIMs train/val/test:',groups.iloc[train].nunique(),groups.iloc[val].nunique(),groups.iloc[test].nunique(),'positive train/val/test:',y.iloc[train].sum(),y.iloc[val].sum(),y.iloc[test].sum())

In [ ]:
def fit_hgb(params):
    model=make_pipeline(SimpleImputer(strategy='median'),HistGradientBoostingClassifier(random_state=SEED,**params))
    weights=compute_sample_weight(class_weight='balanced',y=y.iloc[train])
    model.fit(X.iloc[train],y.iloc[train],histgradientboostingclassifier__sample_weight=weights)
    return model
def best_threshold(actual,prob):
    ts=np.linspace(.01,.99,99); scores=[f1_score(actual,(prob>=t).astype(int),zero_division=0) for t in ts]; i=int(np.argmax(scores)); return float(ts[i]),float(scores[i])
param_grid={'learning_rate':[.04,.08],'max_leaf_nodes':[7,15,31],'max_iter':[150,250],'l2_regularization':[0.0,1.0]}
tuning=[]; best=None
for params in ParameterGrid(param_grid):
    model=fit_hgb(params); prob=model.predict_proba(X.iloc[val])[:,1]; threshold,score=best_threshold(y.iloc[val],prob)
    tuning.append({**params,'validation_f1':score,'threshold':threshold})
    if best is None or score>best[0]: best=(score,threshold,params,model)
tuning_df=pd.DataFrame(tuning).sort_values('validation_f1',ascending=False); tuning_df.to_csv(OUT/'tuning_results.csv',index=False); print(tuning_df.head(10))

In [ ]:
val_score,threshold,best_params,model=best
test_prob=model.predict_proba(X.iloc[test])[:,1]; test_pred=(test_prob>=threshold).astype(int); actual=y.iloc[test].to_numpy(); cm=confusion_matrix(actual,test_pred,labels=[0,1])
result={'dataset':'history_SIM_hour','model':'Tuned HistGradientBoosting','threshold':threshold,'validation_f1':val_score,'test_precision':precision_score(actual,test_pred,zero_division=0),'test_recall':recall_score(actual,test_pred,zero_division=0),'test_f1':f1_score(actual,test_pred,zero_division=0),'test_pr_auc':average_precision_score(actual,test_prob),'test_roc_auc':roc_auc_score(actual,test_prob),'tn':cm[0,0],'fp':cm[0,1],'fn':cm[1,0],'tp':cm[1,1],'features':len(columns),**{f'param_{k}':v for k,v in best_params.items()}}
pd.DataFrame([result]).to_csv(OUT/'advanced_model_metrics.csv',index=False)
pd.DataFrame(cm,index=['actual_legitimate','actual_fraudulent'],columns=['predicted_legitimate','predicted_fraudulent']).to_csv(OUT/'advanced_confusion_matrix.csv')
pd.DataFrame({'sim':groups.iloc[test].to_numpy(),'actual':actual,'probability':test_prob,'predicted':test_pred}).to_csv(OUT/'advanced_test_predictions.csv',index=False)
print(result); print(pd.DataFrame(cm,index=['actual_legitimate','actual_fraudulent'],columns=['predicted_legitimate','predicted_fraudulent']))

Compare `advanced_model_metrics.csv` with the baseline `baseline_metrics.csv`. The advanced model is selected using validation F1; the test set is used once for final reporting. Because the saved history table still contains `static_*` fields, this notebook removes them explicitly before training.